<a href="https://colab.research.google.com/github/lalawee/special-octo-barnacle/blob/main/CartPole_DQN_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:

import random
import gym
import numpy as np
from collections import deque

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

np.bool8 = np.bool_  # temp fix for numpy deprecation

#
# State output is [Pos Velocity Angle Angular-Velocity] for Cartpole
#
env = gym.make('CartPole-v1', new_step_api=True)
print(env.observation_space)
print(env.observation_space.shape)
print(env.observation_space.shape[0], 'State space')
print(env.action_space.n, 'Action space')


Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
(4,)
4 State space
2 Action space


In [7]:
# ==============================================================================
# Helper function to build the NN (used for both primary and target networks)
# ==============================================================================
def build_model():
    model = Sequential()
    model.add(Dense(32, input_dim=env.observation_space.shape[0], activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(env.action_space.n, activation='linear'))
    model.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0), loss='mse')
    return model

# ==============================================================================
# Create Primary Network and Target Network
# ==============================================================================
model = build_model()       # Primary network (used for action selection & training)
target_NN = build_model()   # Target network (used for computing TD targets)

# Initialize target network weights to match the primary network
target_NN.set_weights(model.get_weights())

In [8]:
# ==============================================================================
# Hyper Parameters
# ==============================================================================
GAMMA = 0.99
EPSILON = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
MEMORYSIZE = 100000
BATCHSIZE = 64
C = 1000  # Sync target network weights every C steps

memory = deque(maxlen=MEMORYSIZE)
score = deque(maxlen=100)


In [9]:
# ==============================================================================
# Function to copy weights from primary network to target network
# ==============================================================================
def hard_update_target_model():
    model_wts = model.get_weights()
    target_NN.set_weights(model_wts)

# ==============================================================================
# Experience Replay (Double DQN version)
# ==============================================================================
def experience_replay():
    if len(memory) < BATCHSIZE:
        return

    batch = random.sample(memory, BATCHSIZE)
    states = np.array([s[0] for s, a, r, s_, done in batch])
    next_states = np.array([s_[0] for s, a, r, s_, done in batch])

    # Batch predict — 3 calls total instead of 192
    q_values = model.predict(states, verbose=0)
    q_next_primary = model.predict(next_states, verbose=0)
    q_next_target = target_NN.predict(next_states, verbose=0)

    for idx, (s, a, r, s_, done) in enumerate(batch):
        if not done:
            best_action = np.argmax(q_next_primary[idx])
            target = r + GAMMA * q_next_target[idx][best_action]
        else:
            target = r
        q_values[idx][a] = target

    model.fit(states, q_values, epochs=1, batch_size=BATCHSIZE, verbose=0)


In [10]:
# ======================================================================
# PHASE 1: TRAINING
# Stop when reward per episode > 200
# ======================================================================
print("\n========== PHASE 1: TRAINING ==========\n")

i = 0          # Episode counter
step_count = 0 # Global step counter (for target network sync)
eps = EPSILON  # Start fresh with full exploration

while True:
    i += 1
    state = np.array([env.reset()])  # [[Pos Velocity Angle Angular-Velocity]]
    reward_per_episode = 0

    for t in range(500):
        # Epsilon-greedy policy (using primary network)
        if np.random.rand() <= eps:
            action = env.action_space.sample()
        else:
            action = np.argmax(model.predict(state, verbose=0))

        # Step to next state
        state_, reward, done, _, _ = env.step(action)
        next_state = np.array([state_])

        reward_per_episode += reward

        # Store experience in replay memory
        memory.append((state, action, reward, next_state, done))

        step_count += 1

        if done:
            break

        state = next_state

        # Train primary network via experience replay
        experience_replay()

        # Hard update: sync target network every C steps
        if step_count % C == 0:
            hard_update_target_model()
            print(f"  >> Target network synced at step {step_count}")

    # Epsilon decay
    if eps > EPSILON_MIN:
        eps *= EPSILON_DECAY

    # Record score
    score.append(reward_per_episode)

    # Print episode summary
    print(f"Episode: {i}, Score: {reward_per_episode}, Avg over last 100: {np.mean(score):.1f}, eps={eps:.3f}")

    # Stop training when reward per episode > 200
    if reward_per_episode > 200:
        print(f"\nTraining stopped at Episode {i} with reward {reward_per_episode} > 200")
        break



========== PHASE 1: TRAINING ==========

Episode: 1, Score: 30.0, Avg over last 100: 30.0, eps=0.995
Episode: 2, Score: 15.0, Avg over last 100: 22.5, eps=0.990
Episode: 3, Score: 42.0, Avg over last 100: 29.0, eps=0.985
Episode: 4, Score: 23.0, Avg over last 100: 27.5, eps=0.980
Episode: 5, Score: 19.0, Avg over last 100: 25.8, eps=0.975
Episode: 6, Score: 11.0, Avg over last 100: 23.3, eps=0.970
Episode: 7, Score: 21.0, Avg over last 100: 23.0, eps=0.966
Episode: 8, Score: 43.0, Avg over last 100: 25.5, eps=0.961
Episode: 9, Score: 39.0, Avg over last 100: 27.0, eps=0.956
Episode: 10, Score: 22.0, Avg over last 100: 26.5, eps=0.951
Episode: 11, Score: 15.0, Avg over last 100: 25.5, eps=0.946
Episode: 12, Score: 12.0, Avg over last 100: 24.3, eps=0.942
Episode: 13, Score: 15.0, Avg over last 100: 23.6, eps=0.937
Episode: 14, Score: 22.0, Avg over last 100: 23.5, eps=0.932
Episode: 15, Score: 23.0, Avg over last 100: 23.5, eps=0.928
Episode: 16, Score: 30.0, Avg over last 100: 23.9, e

In [11]:
memory = deque(maxlen=MEMORYSIZE) #using deque (FIFO) for experience-replay memory
score = deque(maxlen=100) #keep track of scores of consecutive 100 episodes as defined in the winning requirement of Cartpole

In [12]:
# ======================================================================
# PHASE 2: EVALUATION (100 episodes, no training, no exploration)
# Print rewards every 10 episodes
# ======================================================================
print("\n========== PHASE 2: EVALUATION (100 Episodes) ==========\n")

eval_rewards = []

for ep in range(1, 101):
    state = np.array([env.reset()])
    reward_per_episode = 0

    for t in range(500):
        # Greedy policy only (no exploration, eps=0)
        action = np.argmax(model.predict(state, verbose=0))

        state_, reward, done, _, _ = env.step(action)
        next_state = np.array([state_])

        reward_per_episode += reward

        if done:
            break

        state = next_state

    eval_rewards.append(reward_per_episode)

    # Print every 10 episodes
    if ep % 10 == 0:
        avg_last_10 = np.mean(eval_rewards[-10:])
        print(f"Eval Episode: {ep}, Score: {reward_per_episode}, Avg last 10: {avg_last_10:.1f}")

# Final evaluation summary
print(f"\n===== EVALUATION RESULTS =====")
print(f"Average reward over 100 episodes: {np.mean(eval_rewards):.1f}")
print(f"Min: {np.min(eval_rewards):.0f}, Max: {np.max(eval_rewards):.0f}")



========== PHASE 2: EVALUATION (100 Episodes) ==========

Eval Episode: 10, Score: 398.0, Avg last 10: 277.4
Eval Episode: 20, Score: 371.0, Avg last 10: 288.6
Eval Episode: 30, Score: 224.0, Avg last 10: 249.1
Eval Episode: 40, Score: 337.0, Avg last 10: 268.7
Eval Episode: 50, Score: 271.0, Avg last 10: 245.4
Eval Episode: 60, Score: 376.0, Avg last 10: 324.8
Eval Episode: 70, Score: 293.0, Avg last 10: 286.1
Eval Episode: 80, Score: 228.0, Avg last 10: 256.1
Eval Episode: 90, Score: 409.0, Avg last 10: 294.8
Eval Episode: 100, Score: 314.0, Avg last 10: 281.6

===== EVALUATION RESULTS =====
Average reward over 100 episodes: 277.3
Min: 189, Max: 493
